### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mic",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/579/myocardial+infarction+complications",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/mic/ && wget -P local-data-warehouse/mic/ https://archive.ics.uci.edu/static/public/579/myocardial+infarction+complications.zip && unzip local-data-warehouse/mic/myocardial+infarction+complications.zip -d local-data-warehouse/mic/ && rm local-data-warehouse/mic/myocardial+infarction+complications.zip
""",

# References
academic_reference_bibtex="""@article{golovenkin2020trajectories,
  title={Trajectories, bifurcations, and pseudo-time in large clinical datasets: applications to myocardial infarction and diabetes data},
  author={Golovenkin, Sergey E and Bac, Jonathan and Chervov, Alexander and Mirkes, Evgeny M and Orlova, Yuliya V and Barillot, Emmanuel and Gorban, Alexander N and Zinovyev, Andrei},
  journal={GigaScience},
  volume={9},
  number={11},
  pages={giaa128},
  year={2020},
  publisher={Oxford University Press}
}
""",
    academic_reference_bibtex_key="golovenkin2020trajectories",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- There are 12 possible targets and four possible time moments to predict the targets for this dataset. We use the categorical target as it summarizes the other possible targets. Furthermore, we use the last possible time point for prediction to utilize all available data.
- We treat "?" as missing values.
- We dropped the "id" column.
- We reversed the ordinal encoding of the target feature.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LET_IS",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="LET_IS",
)

## Preprocessing

In [2]:
import pandas as pd


df = pd.read_csv(f"{dataset_mold.path}/MI.data", na_values="?")
df.columns = [
    "ID", "AGE", "SEX", "INF_ANAM", "STENOK_AN", "FK_STENOK", "IBS_POST", "IBS_NASL", "GB", "SIM_GIPERT", "DLIT_AG", "ZSN_A", "nr_11", "nr_01", "nr_02", "nr_03", "nr_04", "nr_07", "nr_08", "np_01", "np_04", "np_05", "np_07", "np_08", "np_09", "np_10", "endocr_01", "endocr_02", "endocr_03", "zab_leg_01", "zab_leg_02", "zab_leg_03", "zab_leg_04", "zab_leg_06", "S_AD_KBRIG", "D_AD_KBRIG", "S_AD_ORIT", "D_AD_ORIT", "O_L_POST", "K_SH_POST", "MP_TP_POST", "SVT_POST", "GT_POST", "FIB_G_POST", "ant_im", "lat_im", "inf_im", "post_im", "IM_PG_P", "ritm_ecg_p_01", "ritm_ecg_p_02", "ritm_ecg_p_04", "ritm_ecg_p_06", "ritm_ecg_p_07", "ritm_ecg_p_08", "n_r_ecg_p_01", "n_r_ecg_p_02", "n_r_ecg_p_03", "n_r_ecg_p_04", "n_r_ecg_p_05", "n_r_ecg_p_06", "n_r_ecg_p_08", "n_r_ecg_p_09", "n_r_ecg_p_10", "n_p_ecg_p_01", "n_p_ecg_p_03", "n_p_ecg_p_04", "n_p_ecg_p_05", "n_p_ecg_p_06", "n_p_ecg_p_07", "n_p_ecg_p_08", "n_p_ecg_p_09", "n_p_ecg_p_10", "n_p_ecg_p_11", "n_p_ecg_p_12", "fibr_ter_01", "fibr_ter_02", "fibr_ter_03", "fibr_ter_05", "fibr_ter_06", "fibr_ter_07", "fibr_ter_08", "GIPO_K", "K_BLOOD", "GIPER_NA", "NA_BLOOD", "ALT_BLOOD", "AST_BLOOD", "KFK_BLOOD", "L_BLOOD", "ROE", "TIME_B_S", "R_AB_1_n", "R_AB_2_n", "R_AB_3_n", "NA_KB", "NOT_NA_KB", "LID_KB", "NITR_S", "NA_R_1_n", "NA_R_2_n", "NA_R_3_n", "NOT_NA_1_n", "NOT_NA_2_n", "NOT_NA_3_n", "LID_S_n", "B_BLOK_S_n", "ANT_CA_S_n", "GEPAR_S_n", "ASP_S_n", "TIKL_S_n", "TRENT_S_n", "FIBR_PREDS", "PREDS_TAH", "JELUD_TAH", "FIBR_JELUD", "A_V_BLOK", "OTEK_LANC", "RAZRIV", "DRESSLER", "ZSN", "REC_IM", "P_IM_STEN", "LET_IS",
]
df = df.drop(columns=[
    "ID", 'FIBR_PREDS', 'PREDS_TAH', 'JELUD_TAH', 'FIBR_JELUD', 'A_V_BLOK',
    'OTEK_LANC', 'RAZRIV', 'DRESSLER', 'ZSN', 'REC_IM', 'P_IM_STEN'
])

target_feature = "LET_IS"
df[target_feature] = df[target_feature].map(
    {
        1: "cardiogenic_shock",
        0: "alive",
        2: "pulmonary_edema",
        3: "myocardial_rupture",
        4: "progress_congestive_heart_failure",
        5: "thromboembolism",
        6: "asystole",
        7: "ventricular_fibrillation",
    }
)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)


cat_features = [
"SEX","INF_ANAM","STENOK_AN","FK_STENOK","IBS_POST","IBS_NASL","GB","SIM_GIPERT","DLIT_AG","ZSN_A","nr_11","nr_01","nr_02","nr_03","nr_04","nr_07","nr_08","np_01","np_04","np_05","np_07","np_08","np_09","np_10","endocr_01","endocr_02","endocr_03","zab_leg_01","zab_leg_02","zab_leg_03","zab_leg_04","zab_leg_06","O_L_POST","K_SH_POST","MP_TP_POST","SVT_POST","GT_POST","FIB_G_POST","ant_im","lat_im","inf_im","post_im","IM_PG_P","ritm_ecg_p_01","ritm_ecg_p_02","ritm_ecg_p_04","ritm_ecg_p_06","ritm_ecg_p_07","ritm_ecg_p_08","n_r_ecg_p_01","n_r_ecg_p_02","n_r_ecg_p_03","n_r_ecg_p_04","n_r_ecg_p_05","n_r_ecg_p_06","n_r_ecg_p_08","n_r_ecg_p_09","n_r_ecg_p_10","n_p_ecg_p_01","n_p_ecg_p_03","n_p_ecg_p_04","n_p_ecg_p_05","n_p_ecg_p_06","n_p_ecg_p_07","n_p_ecg_p_08","n_p_ecg_p_09","n_p_ecg_p_10","n_p_ecg_p_11","n_p_ecg_p_12","fibr_ter_01","fibr_ter_02","fibr_ter_03","fibr_ter_05","fibr_ter_06","fibr_ter_07","fibr_ter_08","GIPO_K","GIPER_NA","TIME_B_S","R_AB_1_n","R_AB_2_n","R_AB_3_n","NA_KB","NOT_NA_KB","LID_KB","NITR_S","NOT_NA_1_n","LID_S_n","B_BLOK_S_n","ANT_CA_S_n","GEPAR_S_n","ASP_S_n","TIKL_S_n","TRENT_S_n",
    "LET_IS"
]

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,699
Columns: 112
Use sampling: False (sample size: 1,699)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['L_BLOOD', 'ALT_BLOOD', 'AGE', 'AST_BLOOD', 'ROE', 'K_BLOOD', 'NA_BLOOD', 'S_AD_ORIT', 'S_AD_KBRIG', 'D_AD_KBRIG']
Rows remaining as candidates after top-10 filter: 8 (of 1,699)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,DLIT_AG,ZSN_A,nr_11,nr_01,nr_02,nr_03,nr_04,nr_07,nr_08,np_01,np_04,np_05,np_07,np_08,np_09,np_10,endocr_01,endocr_02,endocr_03,zab_leg_01,zab_leg_02,zab_leg_03,zab_leg_04,zab_leg_06,S_AD_KBRIG,D_AD_KBRIG,S_AD_ORIT,D_AD_ORIT,O_L_POST,K_SH_POST,MP_TP_POST,SVT_POST,GT_POST,FIB_G_POST,ant_im,lat_im,inf_im,post_im,IM_PG_P,ritm_ecg_p_01,ritm_ecg_p_02,ritm_ecg_p_04,ritm_ecg_p_06,ritm_ecg_p_07,ritm_ecg_p_08,n_r_ecg_p_01,n_r_ecg_p_02,n_r_ecg_p_03,n_r_ecg_p_04,n_r_ecg_p_05,n_r_ecg_p_06,n_r_ecg_p_08,n_r_ecg_p_09,n_r_ecg_p_10,n_p_ecg_p_01,n_p_ecg_p_03,n_p_ecg_p_04,n_p_ecg_p_05,n_p_ecg_p_06,n_p_ecg_p_07,n_p_ecg_p_08,n_p_ecg_p_09,n_p_ecg_p_10,n_p_ecg_p_11,n_p_ecg_p_12,fibr_ter_01,fibr_ter_02,fibr_ter_03,fibr_ter_05,fibr_ter_06,fibr_ter_07,fibr_ter_08,GIPO_K,K_BLOOD,GIPER_NA,NA_BLOOD,ALT_BLOOD,AST_BLOOD,KFK_BLOOD,L_BLOOD,ROE,TIME_B_S,R_AB_1_n,R_AB_2_n,R_AB_3_n,NA_KB,NOT_NA_KB,LID_KB,NITR_S,NA_R_1_n,NA_R_2_n,NA_R_3_n,NOT_NA_1_n,NOT_NA_2_n,NOT_NA_3_n,LID_S_n,B_BLOK_S_n,ANT_CA_S_n,GEPAR_S_n,ASP_S_n,TIKL_S_n,TRENT_S_n,LET_IS
0,55.0,1,1.0,2.0,2.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,70.0,130.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.6,0.0,132.0,0.30,0.15,NaN,8.9,10.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,alive
1,64.0,0,0.0,4.0,2.0,1.0,NaN,3.0,0.0,7.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.2,0.0,136.0,0.15,0.41,NaN,8.1,5.0,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,alive
2,78.0,1,1.0,2.0,2.0,1.0,NaN,3.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.23,0.07,NaN,5.6,5.0,2.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,alive
3,61.0,1,0.0,0.0,0.0,2.0,NaN,2.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,100.0,70.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.5,7.0,5.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,alive
4,78.0,1,0.0,4.0,2.0,2.0,NaN,2.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,120.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.8,0.0,139.0,0.23,0.15,NaN,6.2,7.0,2.0,1.0,1.0,0.0,NaN,NaN,NaN,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,alive


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,IBS_NASL,category,1627.0,95.76,2.0,"0.0, 1.0"
1,NOT_NA_KB,category,685.0,40.32,2.0,"1.0, 0.0"
2,LID_KB,category,676.0,39.79,2.0,"0.0, 1.0"
3,NA_KB,category,656.0,38.61,2.0,"1.0, 0.0"
4,GIPER_NA,category,375.0,22.07,2.0,"0.0, 1.0"
5,GIPO_K,category,369.0,21.72,2.0,"0.0, 1.0"
6,DLIT_AG,category,248.0,14.60,8.0,"0.0, 7.0, 6.0, 1.0, 5.0, 2.0, 3.0, 4.0"
7,ritm_ecg_p_01,category,152.0,8.95,2.0,"1.0, 0.0"
8,ritm_ecg_p_02,category,152.0,8.95,2.0,"0.0, 1.0"
9,ritm_ecg_p_04,category,152.0,8.95,2.0,"0.0, 1.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
AGE,1691.0,61.848019,11.257238,26.00,92.00
S_AD_KBRIG,624.0,136.907051,34.997835,0.00,260.00
D_AD_KBRIG,624.0,81.394231,19.745045,0.00,190.00
S_AD_ORIT,1432.0,134.556564,31.336338,0.00,260.00
D_AD_ORIT,1432.0,82.737430,18.321785,0.00,190.00
K_BLOOD,1328.0,4.191039,0.754231,2.30,8.20
NA_BLOOD,1324.0,136.549849,6.514459,117.00,169.00
ALT_BLOOD,1416.0,0.481455,0.387261,0.03,3.00
AST_BLOOD,1415.0,0.263717,0.201802,0.04,2.15
KFK_BLOOD,4.0,2.000000,1.095445,1.20,3.60


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                                        
ANT_CA_S_n    1                          1.0   1125  66.22
              2                          0.0    561  33.02
              3                         <NA>     13   0.77
ASP_S_n       1                          1.0   1251  73.63
              2                          0.0    431  25.37
              3                         <NA>     17   1.00
B_BLOK_S_n    1                          0.0   1473  86.70
              2                          1.0    215  12.65
              3                         <NA>     11   0.65
DLIT_AG       1                          0.0    551  32.43
              2                          7.0    431  25.37
              3                         <NA>    248  14.60
              4                          6.0    165   9.71
              5                          1.0     93   5.47
FIB_G_POST    1                          0.0   1672  98.41
              2                          1.0     15   0.88
              3                         <NA>     12   0.71
FK_STENOK     1                          2.0    854  50.26
              2                          0.0    661  38.91
              3                         <NA>     73   4.30
              4                          3.0     54   3.18
              5                          1.0     46   2.71
GB            1                          2.0    880  51.80
              2                          0.0    605  35.61
              3                          3.0    194  11.42
              4                          1.0     11   0.65
              5                         <NA>      9   0.53
GEPAR_S_n     1                          1.0   1202  70.75
              2                          0.0    480  28.25
              3                         <NA>     17   1.00
GIPER_NA      1                          0.0   1294  76.16
              2                         <NA>    375  22.07
              3                          1.0     30   1.77
GIPO_K        1                          0.0    796  46.85
              2                          1.0    534  31.43
              3                         <NA>    369  21.72
GT_POST       1                          0.0   1679  98.82
              2                         <NA>     12   0.71
              3                          1.0      8   0.47
IBS_NASL      1                         <NA>   1627  95.76
              2                          0.0     45   2.65
              3                          1.0     27   1.59
IBS_POST      1                          2.0    682  40.14
              2                          1.0    548  32.25
              3                          0.0    418  24.60
              4                         <NA>     51   3.00
IM_PG_P       1                          0.0   1648  97.00
              2                          1.0     50   2.94
              3                         <NA>      1   0.06
INF_ANAM      1                          0.0   1060  62.39
              2                          1.0    410  24.13
              3                          2.0    146   8.59
              4                          3.0     79   4.65
              5                         <NA>      4   0.24
K_SH_POST     1                          0.0   1638  96.41
              2                          1.0     46   2.71
              3                         <NA>     15   0.88
LET_IS        1                        alive   1428  84.05
              2            cardiogenic_shock    110   6.47
              3           myocardial_rupture     54   3.18
              4                     asystole     27   1.59
              5     ventricular_fibrillation     27   1.59
LID_KB        1                         <NA>    676  39.79
              2                          0.0    627  36.90
              3                          1.0    396  23.31
LID_S_n       1                          0.0   1211  71.28
              2            

In [8]:
# Target Distribution
target_df

,count,pct
LET_IS,,
alive,1428,84.05
cardiogenic_shock,110,6.47
myocardial_rupture,54,3.18
asystole,27,1.59
ventricular_fibrillation,27,1.59
progress_congestive_heart_failure,23,1.35
pulmonary_edema,18,1.06
thromboembolism,12,0.71


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to mic/019d5ce5-2c20-7057-8545-8e291f3bd5e2
019d5ce5-2c20-7057-8545-8e291f3bd5e2
b2153a8bd5e25e79eb720fdb0231245825e38a8857da6161c09c2bdee83dcc42
